In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#


In [2]:
# Env: routesim

import pandas as pd

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem


In [3]:
FNAME_IN_RETROSTAR_TARGETS = '../data/input/retrostar_raw_test.csv'

FNAME_IN_SIMPRETRO_TARGETS = '../data/input/SMILES.txt'

FNAME_OUT = '../data/output/tms.tsv'

RND_SEED = 111142111

TEST_MODE_ON = False

SAMPLE_SIZE = 200

In [4]:
def smiles_to_inchikey(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        try:
            return (AllChem.MolToInchiKey(mol))
        except Exception:
            return ('IKERROR')
    return ('IKERROR')


In [5]:

df_simpretro = pd.read_csv (FNAME_IN_SIMPRETRO_TARGETS, sep = '\t', header = None)
df_simpretro.columns = ['smiles']
print (df_simpretro.head)

# These are RXSMILES
df_retrostar = pd.read_csv (FNAME_IN_RETROSTAR_TARGETS, sep = ',')
print (df_retrostar.head)

<bound method NDFrame.head of                                                 smiles
0    Cc1c(Cl)c2c(Cl)c(C)c1-c1c(-c3ccc(F)cc3)sc3ncnc...
1    CC(C)CCN[C@@H]1CCc2cc(O)c(N3CC(=O)NS3(=O)=O)c(...
2       CNC(=O)COc1cc(Cl)c(Cc2ccc(O)c(C(C)C)c2)c(Cl)c1
3    Cc1ncsc1-c1ccc([C@@H](CO)NC(=O)[C@H]2C[C@H](O)...
4    Cc1ncn(CC(=O)N2CCN(c3sc(C(F)(F)F)nc3-c3cnc(C(F...
..                                                 ...
167  Cn1ncc(-c2ccc3c(=O)[nH]nc(CN)c3c2)c1-c1c(F)c(C...
168  Cc1cnc2c(-c3ccc(C(=O)N(c4ncccc4C)[C@@H]4CCCNC4...
169  C=C(C)[C@@H]1CC[C@]2(NCCN3CCS(=O)(=O)CC3)CC[C@...
170  CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...
171  Cn1cnc(Cn2c(=O)[nH]/c(=N\c3cc4cn(C)nc4cc3Cl)n(...

[172 rows x 1 columns]>
<bound method NDFrame.head of              id                                         rxn_smiles  \
0       4070373  [Cl:14][C:15]1[CH:20]=[CH:19][C:18]([C:21](=O)...   
1       1019017  [CH:13]([Mg]Cl)([CH3:15])[CH3:14].[OH:1][C:2]1...   
2        836475  [Cl:1][C:2]1[CH:7]=[CH:6][CH

In [6]:
def extract_product (rxsmiles):
    smiles = rxsmiles.split('>')[-1].               \
                                    split('.')[-1]. \
                                    strip()

    return (smiles)


if TEST_MODE_ON:
    df_retrostar = df_retrostar.sample(n=SAMPLE_SIZE, random_state=RND_SEED)

df_retrostar['tm'] = df_retrostar.apply(lambda x: extract_product(x['cano_rxn_smiles']), axis = 1)



#print (len(list(df['tm'])))
#print (len(set(list(df['tm']))))

df_retrostar = df_retrostar[['tm']].copy()
df_retrostar = df_retrostar.rename (columns = {
    'tm': 'smiles'
})

#df_retrostar.to_csv (FNAME_OUT, sep = '\t', index = False)


df_retrostar_sub = df_retrostar.sample(n = SAMPLE_SIZE, random_state=RND_SEED)






In [7]:
# Merge datasets

df = pd.concat([df_retrostar_sub, df_simpretro])

print (df.head)

<bound method NDFrame.head of                                                    smiles
124698  Cc1ccc(N(C(=O)OC(C)(C)C)c2ccc(C(O)c3cn(S(=O)(=...
100696                             Cc1ccccc1C1CCN(C)CC1CO
45272             COC(=O)C(Oc1ccc([N+](=O)[O-])cc1)C(C)=O
112031                  O=C(Nc1noc2ccccc12)c1c(Cl)cccc1Cl
88623          N#Cc1cc(O)c(O)c(C#N)c1Cc1ccc(CC(F)(F)F)cc1
...                                                   ...
167     Cn1ncc(-c2ccc3c(=O)[nH]nc(CN)c3c2)c1-c1c(F)c(C...
168     Cc1cnc2c(-c3ccc(C(=O)N(c4ncccc4C)[C@@H]4CCCNC4...
169     C=C(C)[C@@H]1CC[C@]2(NCCN3CCS(=O)(=O)CC3)CC[C@...
170     CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...
171     Cn1cnc(Cn2c(=O)[nH]/c(=N\c3cc4cn(C)nc4cc3Cl)n(...

[372 rows x 1 columns]>


In [8]:
# Generate InChI-Keys

df['inchikey'] = df.apply(lambda x: smiles_to_inchikey(x['smiles']), axis = 1)

df = df[df['inchikey'] != 'IKERROR'].copy()

print (df.head)



<bound method NDFrame.head of                                                    smiles  \
124698  Cc1ccc(N(C(=O)OC(C)(C)C)c2ccc(C(O)c3cn(S(=O)(=...   
100696                             Cc1ccccc1C1CCN(C)CC1CO   
45272             COC(=O)C(Oc1ccc([N+](=O)[O-])cc1)C(C)=O   
112031                  O=C(Nc1noc2ccccc12)c1c(Cl)cccc1Cl   
88623          N#Cc1cc(O)c(O)c(C#N)c1Cc1ccc(CC(F)(F)F)cc1   
...                                                   ...   
167     Cn1ncc(-c2ccc3c(=O)[nH]nc(CN)c3c2)c1-c1c(F)c(C...   
168     Cc1cnc2c(-c3ccc(C(=O)N(c4ncccc4C)[C@@H]4CCCNC4...   
169     C=C(C)[C@@H]1CC[C@]2(NCCN3CCS(=O)(=O)CC3)CC[C@...   
170     CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...   
171     Cn1cnc(Cn2c(=O)[nH]/c(=N\c3cc4cn(C)nc4cc3Cl)n(...   

                           inchikey  
124698  WPAUDXVOHGJMLA-UHFFFAOYSA-N  
100696  YMGSLNVZHUIQNC-UHFFFAOYSA-N  
45272   BVAJBVZPXNQRIA-UHFFFAOYSA-N  
112031  MXHBMPVWAPEYSH-UHFFFAOYSA-N  
88623   VUEKMKATBGIPOP-UHFFFAOYSA-N  
...      

In [9]:
# Deduplicate

df = df.groupby(['inchikey'], as_index = False).first()

print (df.head)

<bound method NDFrame.head of                         inchikey  \
0    AAZPIQPULVRHOW-SFKJMYEFSA-N   
1    ACMYXPZZKWSEOO-ROUUACIJSA-N   
2    ADROYVWTRVGXHG-UHFFFAOYSA-N   
3    AEMZJBZKICOPPC-DZDWSLRDSA-N   
4    AGOWMJFPYPUNRF-UHFFFAOYSA-N   
..                           ...   
367  ZNTVNNXGPZITBD-UHFFFAOYSA-N   
368  ZQXOMIPSVVBRGV-UHFFFAOYSA-N   
369  ZRBPIAWWRPFDPY-IRXDYDNUSA-N   
370  ZTXVRZLOVRFPEC-UHFFFAOYSA-N   
371  ZVERWTXKKWSSHH-UHFFFAOYSA-N   

                                                smiles  
0    COc1ccc(CO[C@H]2CC[C@@]3(C)C(=CC[C@H]4[C@@H]5C...  
1    CC(C)(C)OC(=O)CC[C@H](NC(=O)OCc1ccccc1)C(=O)N[...  
2                                       COC(CC#N)=NC#N  
3    CC(C)(O)c1ccc(-c2cccc(N(CC34CCC(c5noc(C(C)(C)F...  
4                   Nc1nc(CSc2ccc([N+](=O)[O-])cc2)cs1  
..                                                 ...  
367             COC(=O)c1cn2cc(-c3ccccc3)cc(C(C)C)c2n1  
368           Cn1ccnc1S(=O)Cc1cccc2cc(-c3nccs3)[nH]c12  
369  C=CC(=O)N1CCN(c2nc(

In [10]:
df.to_csv (FNAME_OUT, sep = '\t', index = False)

print ('[Done.]')

[Done.]
